In [11]:
def get_all_partitions(n):
    """Generates all integer partitions of n."""
    def generate(n, max_val):
        if n == 0:
            yield []
        for i in range(min(n, max_val), 0, -1):
            for p in generate(n - i, i):
                yield [i] + p
    return list(generate(n, n))

def are_adjacent(p1, p2):
    """
    Checks if partition p2 can be reached from p1 via the allowed Gray code transition:
    Decreasing one part by 1 and increasing another part by 1 (or adding a new 1).
    """
    # Create frequency maps for the parts in both partitions
    from collections import Counter
    c1, c2 = Counter(p1), Counter(p2)
    
    # Ignore parts that are identical in both partitions
    common = c1 & c2
    c1.subtract(common)
    c2.subtract(common)
    
    # Filter out elements with zero counts
    diff1 = sorted(list(c1.elements()))
    diff2 = sorted(list(c2.elements()))
    
    # The transition means exactly one element goes down by 1, and one goes up by 1.
    # Therefore, the difference between the two partitions must be exactly two parts changing.
    if len(diff1) == 1 and len(diff2) == 2:
        # Case: A part split into (part-1) and 1
        return diff1[0] - 1 == diff2[1] and diff2[0] == 1
    elif len(diff1) == 2 and len(diff2) == 1:
        # Case: (part-1) and 1 merged into a part
        return diff2[0] - 1 == diff1[1] and diff1[0] == 1
    elif len(diff1) == 2 and len(diff2) == 2:
        # Case: One part decreases by 1, another increases by 1
        # Example: {3, 3} -> {4, 2}
        return (diff1[0] + 1 == diff2[1] and diff1[1] - 1 == diff2[0]) or \
               (diff1[1] + 1 == diff2[1] and diff1[0] - 1 == diff2[0])
    
    return False

def find_gray_code_sequence(n):
    """Finds a Hamiltonian path (Gray code sequence) through the partition graph."""
    partitions = get_all_partitions(n)
    
    # Build the adjacency graph
    graph = {i: [] for i in range(len(partitions))}
    for i in range(len(partitions)):
        for j in range(i + 1, len(partitions)):
            if are_adjacent(partitions[i], partitions[j]):
                graph[i].append(j)
                graph[j].append(i)

    # Backtracking function to find the Hamiltonian path
    def dfs(current_node, visited, path):
        if len(path) == len(partitions):
            return path
        
        for neighbor in graph[current_node]:
            if neighbor not in visited:
                visited.add(neighbor)
                path.append(neighbor)
                
                result = dfs(neighbor, visited, path)
                if result:
                    return result
                
                # Backtrack
                visited.remove(neighbor)
                path.pop()
                
        return None

    # The paper notes the enumeration typically starts at the lexicographically largest partition,
    # which is simply [n] itself. 
    start_index = partitions.index([n])
    path = dfs(start_index, {start_index}, [start_index])
    
    if path:
        return [partitions[i] for i in path]
    else:
        return None

# --- Example Usage ---
n = 7
gray_code = find_gray_code_sequence(n)

print(f"Gray Code sequence for integer partitions of {n}:")
for p in gray_code:
    print(p)

Gray Code sequence for integer partitions of 7:


TypeError: 'NoneType' object is not iterable